# Setup and Validation Notebook

Notebook ini untuk:
1. Install dependencies di Colab
2. Upload dataset dari lokal ke Colab runtime
3. Check GPU availability
4. Load dan validate datasets
5. Test tokenizer
6. Setup model dengan LoRA
7. Baseline evaluation

> **Catatan:** Notebook ini dijalankan via VS Code yang terhubung ke Colab kernel.
> Dataset di-upload dari lokal ke Colab runtime (`/content/`).

## 1. Install Dependencies

In [3]:
# Install semua dependencies yang dibutuhkan di Colab runtime
# (berbeda dengan conda env lokal - harus install ulang)
!pip install -q transformers peft datasets accelerate
!pip install -q evaluate rouge_score bert_score
print('✓ Dependencies installed')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00
✓ Dependencies installed


In [7]:
# Cek versi semua library yang terinstall
import importlib
import subprocess
import sys, torch, platform

libs = [
    "transformers",
    "peft", 
    "datasets",
    "accelerate",
    "evaluate",
    "torch",
    "tokenizers",
    "rouge_score",
    "bert_score",
]


print(f"Python:  {sys.version}")
print(f"OS:      {platform.system()}")
print(f"Torch:   {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
print(f"\n ")

print("=== Library Versions ===")
for lib in libs:
    try:
        mod = importlib.import_module(lib.replace("-", "_"))
        version = getattr(mod, "__version__", "unknown")
        print(f"  {lib:<20} {version}")
    except ImportError:
        print(f"  {lib:<20} NOT INSTALLED")

# Cek Python version
import sys
print(f"\n  {'python':<20} {sys.version.split()[0]}")

# Cek CUDA
import torch
print(f"  {'cuda available':<20} {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  {'cuda version':<20} {torch.version.cuda}")
    print(f"  {'gpu name':<20} {torch.cuda.get_device_name(0)}")


Python:  3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
OS:      Linux
Torch:   2.10.0+cu128
CUDA:    True

 
=== Library Versions ===
  transformers         5.0.0
  peft                 0.18.1
  datasets             4.0.0
  accelerate           1.13.0
  evaluate             0.4.6
  torch                2.10.0+cu128
  tokenizers           0.22.2
  rouge_score          unknown
  bert_score           0.3.12

  python               3.12.13
  cuda available       True
  cuda version         12.8
  gpu name             Tesla T4


In [2]:
import tensorflow as tf
tf.test.gpu_device_name()
# Output should indicate a GPU (e.g., '/device:GPU:0')

!nvidia-smi



Fri Apr 10 13:35:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P0             26W /   70W |     105MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload Dataset ke Colab Runtime

Ada 2 opsi untuk akses dataset:
- **Opsi A (Recommended):** Upload dari lokal via `files.upload()` atau drag-drop
- **Opsi B:** Mount Google Drive jika sudah upload ke Drive

In [8]:
# === OPSI A: Upload dari lokal (tanpa Google Drive) ===
# Jalankan cell ini untuk upload file dataset dari komputer lokal

import os

# Buat direktori di Colab runtime
os.makedirs('/content/dataset_aqg/output_domain', exist_ok=True)
os.makedirs('/content/dataset_aqg/dataset-task-spesifc', exist_ok=True)
os.makedirs('/content/src/finetuned', exist_ok=True)

print('Direktori dibuat:')
print('  /content/dataset_aqg/output_domain/')
print('  /content/dataset_aqg/dataset-task-spesifc/')
print()
print('Selanjutnya upload file dataset menggunakan cell berikut.')

Direktori dibuat:
  /content/dataset_aqg/output_domain/
  /content/dataset_aqg/dataset-task-spesifc/

Selanjutnya upload file dataset menggunakan cell berikut.


In [ ]:
from google.colab import drive
import shutil, os

# Mount Google Drive
drive.mount('/content/drive')

# Path file di Google Drive kamu
drive_path = '/content/drive/MyDrive/dataset_aqg/output_domain'

# Pastikan direktori tujuan ada
os.makedirs('/content/dataset_aqg/output_domain', exist_ok=True)

# Salin file ke direktori Colab
files_to_copy = ['train.jsonl', 'validation.jsonl', 'test.jsonl']
for f in files_to_copy:
    src = f'{drive_path}/{f}'
    dst = f'/content/dataset_aqg/output_domain/{f}'
    shutil.copy(src, dst)
    print(f'✓ {f} → /content/dataset_aqg/output_domain/')

print('\nSemua file berhasil disalin!')


Upload file dari: dataset_aqg/output_domain/
Pilih 3 file: train.jsonl, validation.jsonl, test.jsonl


KeyboardInterrupt: 

In [ ]:
# Upload file dataset task-specific
# File ada di: D:\2-Project\AQG\dataset_aqg\dataset-task-spesifc\

print('Upload file dari: dataset_aqg/dataset-task-spesifc/')
print('Pilih 3 file: train.jsonl, validation.jsonl, test.jsonl')
uploaded2 = files.upload()

for filename in uploaded2.keys():
    os.rename(filename, f'/content/dataset_aqg/dataset-task-spesifc/{filename}')
    print(f'✓ {filename} → /content/dataset_aqg/dataset-task-spesifc/')

In [ ]:
# === OPSI B: Mount Google Drive (jika sudah upload ke Drive) ===
# Uncomment jika ingin pakai Drive

# from google.colab import drive
# drive.mount('/content/drive')
# 
# import shutil
# shutil.copytree(
#     '/content/drive/MyDrive/AQG/dataset_aqg',
#     '/content/dataset_aqg'
# )
# print('✓ Dataset copied from Drive')

print('Opsi B (Drive) dinonaktifkan. Gunakan Opsi A di atas.')

In [ ]:
# Upload source code (src/finetuned/)
# Kita perlu upload module Python agar bisa di-import di Colab

import subprocess

# Cara termudah: zip folder src/finetuned dan upload
print('Upload file src_finetuned.zip')
print('Buat zip dulu di lokal:')
print('  Windows: klik kanan folder src/finetuned → Send to → Compressed folder')
print('  Atau via terminal: cd D:\\2-Project\\AQG && python -m zipfile -c src_finetuned.zip src/finetuned')
print()

uploaded_src = files.upload()  # Upload src_finetuned.zip

# Extract zip
import zipfile
for filename in uploaded_src.keys():
    with zipfile.ZipFile(filename, 'r') as z:
        z.extractall('/content/')
    print(f'✓ {filename} extracted to /content/')

In [ ]:
# Setup Python path agar module bisa di-import
import sys
sys.path.insert(0, '/content')

# Set working directory
os.chdir('/content')

# Verify
print('Python path:', sys.path[:3])
print('Working dir:', os.getcwd())
print()

# Test import
try:
    from src.finetuned.data.dataset_loader import DatasetLoader
    print('✓ Module import berhasil')
except ImportError as e:
    print(f'✗ Import error: {e}')
    print('Pastikan src_finetuned.zip sudah di-extract dengan benar')

## 3. Check GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    print('⚠ GPU tidak tersedia!')
    print('Di Colab: Runtime → Change runtime type → T4 GPU')
    print('Di VS Code: Pastikan Colab kernel yang dipilih sudah enable GPU')
else:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'✓ GPU: {gpu_name}')
    print(f'✓ Memory: {gpu_memory:.1f} GB')
    
    if gpu_memory < 14:
        print(f'⚠ Memory {gpu_memory:.1f}GB mungkin kurang. Target: T4 (15GB)')

## 4. Load dan Validate Datasets

In [ ]:
from src.finetuned.data.dataset_loader import DatasetLoader

loader = DatasetLoader()

# Load domain dataset
print('Loading Domain Adaptation Dataset...')
domain_train = loader.load_dataset('/content/dataset_aqg/output_domain/', split='train')
domain_val   = loader.load_dataset('/content/dataset_aqg/output_domain/', split='validation')

print(f'Domain Train: {len(domain_train)} entries')
print(f'Domain Val:   {len(domain_val)} entries')

In [ ]:
# Load task-specific dataset
print('Loading Task-Specific Dataset...')
task_train = loader.load_dataset('/content/dataset_aqg/dataset-task-spesifc/', split='train')
task_val   = loader.load_dataset('/content/dataset_aqg/dataset-task-spesifc/', split='validation')

print(f'Task Train: {len(task_train)} entries')
print(f'Task Val:   {len(task_val)} entries')

In [ ]:
# Validate datasets
print('=== Domain Dataset Validation ===')
domain_val_result = loader.validate_dataset(domain_train)

print('\n=== Task-Specific Dataset Validation ===')
task_val_result = loader.validate_dataset(task_train)

# Preview sample
print('\n=== Sample Domain Entry ===')
print(f"Input:  {domain_train[0]['input'][:150]}...")
print(f"Target: {domain_train[0]['target'][:150]}...")

print('\n=== Sample Task Entry ===')
print(f"Input:  {task_train[0]['input'][:150]}...")
print(f"Target: {task_train[0]['target'][:150]}...")

## 5. Test Tokenizer

In [ ]:
# IndoNanoT5 menggunakan AutoTokenizer (bukan T5Tokenizer)
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained('LazarusNLP/IndoNanoT5-base')
print(f'✓ Tokenizer loaded: {type(tokenizer).__name__}')
print(f'  Vocab size: {tokenizer.vocab_size}')

In [ ]:
from src.finetuned.data.tokenizer_tester import TokenizerTester

tester = TokenizerTester(tokenizer)

# Test markdown handling
print('Testing markdown handling...')
markdown_results = tester.test_markdown_handling()

# Analyze token distribution
print('\nAnalyzing domain dataset token distribution...')
domain_token_stats = loader.analyze_token_distribution(
    domain_train.select(range(min(50, len(domain_train)))),  # Sample 50 untuk speed
    tokenizer,
    max_length=512
)

print('\nAnalyzing task-specific dataset token distribution...')
task_token_stats = loader.analyze_token_distribution(
    task_train.select(range(min(50, len(task_train)))),
    tokenizer,
    max_length=512
)

## 6. Setup Model dengan LoRA

In [ ]:
from src.finetuned.model.model_setup import ModelSetup
from peft import LoraConfig

model_setup = ModelSetup()

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=['q', 'v'],
    bias='none',
    task_type='SEQ_2_SEQ_LM'
)

peft_model, tokenizer = model_setup.setup_model_for_training(
    model_name='LazarusNLP/IndoNanoT5-base',
    lora_config=lora_config
)

In [ ]:
# Test forward pass
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
peft_model = peft_model.to(device)

dummy_input = tokenizer('Test input Python', return_tensors='pt')
dummy_labels = tokenizer('Test output', return_tensors='pt')['input_ids']

dummy_input = {k: v.to(device) for k, v in dummy_input.items()}
dummy_labels = dummy_labels.to(device)

with torch.no_grad():
    outputs = peft_model(**dummy_input, labels=dummy_labels)

print(f'✓ Forward pass berhasil! Loss: {outputs.loss.item():.4f}')

## 7. Baseline Evaluation

In [ ]:
# Generate baseline predictions (10 samples)
print('Generating baseline predictions (10 samples)...')

samples = task_val.select(range(min(10, len(task_val))))
predictions = []
references  = []

for i, sample in enumerate(samples):
    inputs = tokenizer(
        sample['input'],
        return_tensors='pt',
        max_length=512,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        output_ids = peft_model.generate(**inputs, max_length=256, num_beams=2)
    
    prediction = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    predictions.append(prediction)
    references.append(sample['target'])
    
    print(f'Sample {i+1}: {prediction[:80]}...')

# Compute BLEU
from evaluate import load
bleu = load('bleu')
baseline_bleu = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references]
)

print(f'\nBaseline BLEU-4: {baseline_bleu["bleu"]:.4f}')
print('Expected: < 0.15 (sebelum fine-tuning)')

## 8. Summary

In [ ]:
print('\n' + '='*60)
print('VALIDATION SUMMARY')
print('='*60)

gpu_ok = torch.cuda.is_available()
print(f'\n1. GPU: {"✓ " + torch.cuda.get_device_name(0) if gpu_ok else "✗ Tidak tersedia"}')

print(f'\n2. Datasets:')
print(f'   Domain:        {len(domain_train)} train, {len(domain_val)} val')
print(f'   Task-Specific: {len(task_train)} train, {len(task_val)} val')

print(f'\n3. Token Distribution:')
print(f'   Domain exceed 512:        {domain_token_stats["pct_exceeding_limit"]:.1f}%')
print(f'   Task-Specific exceed 512: {task_token_stats["pct_exceeding_limit"]:.1f}%')

print(f'\n4. Model: ✓ IndoNanoT5-base + LoRA loaded')
print(f'\n5. Baseline BLEU-4: {baseline_bleu["bleu"]:.4f}')

print('\n' + '='*60)
if gpu_ok:
    print('✓ VALIDATION COMPLETE - Siap untuk training!')
    print('  Lanjut ke: 02_domain_adaptation.ipynb')
else:
    print('⚠ Enable GPU dulu sebelum training')
print('='*60)